# EDA — AI4I 2020 Predictive Maintenance Dataset

Exploratory analysis of machine sensor data used to train the failure-prediction model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)

df = pd.read_csv("../data/ai4i2020.csv")
df.shape


## Column information

In [ ]:
df.info()

## Statistical summary

In [ ]:
df.describe(include='all')

## Missing values

In [ ]:
df.isnull().sum()

## Duplicate rows

In [ ]:
df.duplicated().sum()

## Target class distribution

Machine failures are rare events — this is an imbalanced classification problem.

In [ ]:
counts = df["Machine failure"].value_counts()
print(counts)
print(f"Failure rate: {df['Machine failure'].mean()*100:.2f}%")

fig, ax = plt.subplots()
counts.plot(kind="bar", ax=ax, color=["#2563eb", "#ef4444"])
ax.set_xticklabels(["Healthy (0)", "Failure (1)"], rotation=0)
ax.set_title("Machine failure class distribution")
plt.show()


## Machine type distribution

In [ ]:
df["Type"].value_counts().plot(kind="bar", color="#0ea5a4")
plt.title("Machine type (quality variant) distribution")
plt.show()


## Feature distributions

In [ ]:
numeric_cols = ["Air temperature [K]", "Process temperature [K]",
                "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, numeric_cols):
    sns.histplot(df[col], kde=True, ax=ax, color="#2563eb")
    ax.set_title(col)
axes.flat[-1].axis("off")
plt.tight_layout()
plt.show()


## Outlier analysis (boxplots)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, col in zip(axes, numeric_cols):
    sns.boxplot(y=df[col], ax=ax, color="#93c5fd")
    ax.set_title(col, fontsize=9)
plt.tight_layout()
plt.show()


## Correlation analysis

In [ ]:
corr = df[numeric_cols + ["Machine failure"]].corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature correlation matrix")
plt.show()


## Failure vs. individual sensor readings

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, numeric_cols):
    sns.boxplot(x="Machine failure", y=col, data=df, ax=ax, palette=["#2563eb", "#ef4444"])
    ax.set_xticklabels(["Healthy", "Failure"])
    ax.set_title(f"{col} vs Machine failure")
axes.flat[-1].axis("off")
plt.tight_layout()
plt.show()


## Observations

- The dataset is strongly imbalanced: machine failures are a small minority
  of rows. Accuracy alone is a misleading metric here — recall/F1/ROC-AUC on
  the failure class matter far more (see `ml/train_model.py`).
- `Torque [Nm]` and `Tool wear [min]` tend to be higher on average in the
  failure class, consistent with overstrain-type failure mechanics.
- `Rotational speed [rpm]` shows more spread in failure cases, reflecting
  both very low (heat-dissipation-risk) and abnormal operating points.
- `Air temperature [K]` and `Process temperature [K]` are highly correlated
  with each other (process temperature is defined relative to air
  temperature), so most of their predictive signal is redundant.
- The failure-type indicator columns (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`) are
  deliberately **excluded** from the model's input features: they directly
  encode why/whether a failure occurred, so including them would leak the
  target and make the "prediction" trivial and unusable in a real
  before-the-fact maintenance setting.
